# dmf research walkthrough — from raw disputes to a selected model

This notebook runs the **`dmf.research` selection harness end-to-end** on small synthetic
dispute data (~2 minutes total), and shows how to read each output — because each step's
output is the *input* to the next:

```
config  →  variable ordering  →  model × k grid  →  marginal gains  →  1-SE selection  →  holdout confirmation
(the experiment)  (which subsets exist)  (how good is each)  (does var k earn its place?)  (which spec ships)  (the honest number)
                                              ↓
                            prediction store: (row_id, y_true, y_score) per fold & holdout
              (recompute / expand any metric post-run  →  derive & justify the production threshold)
```

Run it from the `examples/` folder, top to bottom. Every cell is short on purpose.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dmf import Config, ProductionScorer
from dmf.research import ModelSelectionHarness
from generate_synthetic_disputes import generate_disputes   # lives beside this notebook

# fixed identity colors: one per model, used consistently in every chart below
C = {"logistic": "#3b6fd4", "gbm": "#c8571b"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})

df = generate_disputes(n=4000, seed=11, prevalence=0.10)
print(f"{df.shape[0]:,} disputes, {df.shape[1]-2} candidate columns, "
      f"prevalence {df['is_fraudulent_dispute'].mean():.1%}")
df.head(3)

## Step 1 — the config *is* the experiment

Everything the harness does is declared here: which models compete, how many variables to
sweep (`k_min..k_max`), the split design, the metric, and which segments to audit on the
holdout. Nothing downstream takes arguments — change the experiment by changing this.

We keep it deliberately small: **2 models × k=1..6 × 3-fold CV**, so a human can watch it run.

Two additions worth noticing: `data.id_column` names the record identifier — it keys the
prediction store and is automatically excluded from the candidate features — and
`run.save_predictions: "all"` keeps the row-level scores behind every CV fold and the
holdout. Steps 8–10 live entirely off that store.

In [ ]:
cfg = Config.from_dict({
    "run": {"name": "walkthrough", "output_dir": "./artifacts_demo", "random_state": 7,
            "n_jobs": 1, "verbose": 0, "save_fitted_model": True,
            "save_predictions": "all"},                      # keep every row-level score
    "data": {"target": "is_fraudulent_dispute", "id_column": "dispute_id"},
    "columns": {"auto_infer": True, "drop": ["merchant_id"]},
    "split": {"holdout_size": 0.25, "cv": {"n_splits": 3}},
    "selection": {"k_min": 1, "k_max": 6, "top_n": 3},
    "metrics": {"primary": "average_precision",
                "secondary": ["roc_auc", "ks_statistic", "brier_score"],
                "slice_columns": ["customer_segment", "claim_channel"], "min_slice_n": 40},
    "models": {
        "logistic": {"estimator": "sklearn.linear_model.LogisticRegression", "tag": "champion",
                     "family": "linear", "requires_scaling": True, "imbalance": "balanced",
                     "params": {"C": 1.0, "max_iter": 1000}},
        "gbm": {"estimator": "sklearn.ensemble.HistGradientBoostingClassifier", "tag": "challenger",
                "family": "tree", "requires_scaling": False, "imbalance": "balanced",
                "params": {"max_iter": 120, "learning_rate": 0.08, "early_stopping": False}},
    },
})
print("models:", list(cfg.enabled_models), "| primary metric:", cfg.metrics.primary,
      "| k sweep: 1..", cfg.selection.k_max, "| prediction store:", cfg.run.save_predictions)

## Step 2 — run the harness once

One call does the whole sequence. Internally, per CV fold it: types the columns → ranks the
variables *inside that fold* (so no validation row helps choose the features it scores) →
fits every model at every k → scores the fold. Then it aggregates, tests marginal gains,
applies the 1-SE rule, and confirms the winner on the untouched 25% holdout.

The step log below is the run's audit trail — one line of numbers per step. Skim it now;
we unpack each step next.

In [ ]:
result = ModelSelectionHarness(cfg).run(df.copy())
print(result.report.render(max_width=130))

## Step 3 — variable ordering & stability

**What it is:** inside each fold, each model ranks all candidate variables; the top-k prefix
of that ranking defines which variable subsets the grid evaluates. **Why it matters
downstream:** the grid can only score subsets the ordering proposes — an unstable ordering
means the "k=4 model" is a different 4 variables in every fold.

**How to read it:** the chart shows the share of folds in which each variable survived into
the top-k. Bars at 1.0 = the backbone of the model, selected every time. Bars near 0.3 =
fold-dependent noise — be suspicious if one of these ends up in the final spec.

In [ ]:
stab = result.report.get("selection_stability")["selection_frequency"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
for ax, (model, freqs) in zip(axes, stab.items()):
    s = pd.Series(freqs).sort_values()
    ax.barh(s.index, s.values, color=C[model], height=0.6)
    ax.set_title(f"{model} — top-k selection frequency across folds", fontsize=10)
    ax.set_xlim(0, 1.05)
    for y, v in enumerate(s.values):
        ax.text(v + 0.02, y, f"{v:.2f}", va="center", fontsize=8, color="#555")
plt.tight_layout()
always = result.report.get("selection_stability")["always_selected"]
print("selected in every fold:", always)

## Step 4 — the leaderboard: performance vs number of variables

**What it is:** every (model × k) cell scored by fold-nested CV. **How to read it:** each
curve is one model; the error bars are the **Nadeau–Bengio corrected** standard error
(folds share training data, so the naive SE would be ~⅓ too narrow). Look for where each
curve *plateaus* — variables past the bend are candidates for cutting, which is exactly
what the next two steps decide formally.

This table is the input to both the marginal-gain tests and the 1-SE selection.

In [ ]:
lb = result.leaderboard
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for model, g in lb.groupby("model"):
    g = g.sort_values("k")
    ax.errorbar(g["k"], g["cv_average_precision_mean"], yerr=g["cv_average_precision_se"],
                color=C[model], marker="o", ms=5, lw=2, capsize=3, label=model)
ax.set_xlabel("k (number of variables)"); ax.set_ylabel("CV average precision")
ax.set_title("Leaderboard: OOS performance vs variable count (NB-corrected SE)", fontsize=10)
ax.legend(frameon=False)
plt.tight_layout()
lb[["rank", "model", "k", "cv_average_precision_mean", "cv_average_precision_se",
    "overfit_gap"]].head(6)

**Also check `overfit_gap`** (train − OOS score): a cell with strong CV performance *and* a
large gap is memorising — the selection rule below only sees the OOS number, so the gap is
your independent sanity check on any spec you're about to trust.

## Step 5 — marginal gains: does the k-th variable earn its place?

**What it is:** for the best model, the paired fold-level change in AP going from k−1 → k
variables, with a Nadeau–Bengio corrected t-test. **How to read it:** solid bars are
statistically significant improvements; faded bars are noise. This is the plateau from
Step 4 made precise — and it's the evidence the 1-SE rule (next) uses implicitly: once
gains stop being significant, extra variables are cost without benefit.

In [ ]:
best_model = result.selected["model"]
g = result.marginal_gains.query("model == @best_model").sort_values("to_k")
fig, ax = plt.subplots(figsize=(7.5, 3.4))
from matplotlib.colors import to_rgba
bar_colors = [to_rgba(C[best_model], 1.0 if v == "improves" else 0.35) for v in g["verdict"]]
ax.bar(g["to_k"], g["mean_delta"], color=bar_colors, width=0.6)
ax.axhline(0, color="#888", lw=1)
for _, r in g.iterrows():
    ax.text(r["to_k"], r["mean_delta"], f' {r["added_variable"]}\n {r["verdict"]}',
            fontsize=7, rotation=90, va="bottom" if r["mean_delta"] >= 0 else "top", ha="center")
ax.set_xlabel("k (variable added at this step)"); ax.set_ylabel("Δ average precision")
ax.set_title(f"{best_model}: marginal value of each added variable (solid = significant)", fontsize=10)
plt.tight_layout()
g[["from_k", "to_k", "added_variable", "mean_delta", "p_value", "verdict"]]

## Step 6 — selection: the one-standard-error rule

**What it is:** among all specs within one (corrected) SE of the outright best, take the one
with the **fewest variables**. Simpler models are cheaper to monitor, explain, and defend —
and within one SE, "best" is statistically indistinguishable from "simplest".

**How to read the chart:** the shaded band is [best − 1 SE, best]. Any point inside the band
is a legitimate choice; the rule picks the leftmost. `cost_of_parsimony` quantifies exactly
how much headline AP was traded for the smaller spec.

In [ ]:
sel = result.selected
gb = lb.query("model == @best_model").sort_values("k")
best = gb["cv_average_precision_mean"].max()
se = gb.loc[gb["cv_average_precision_mean"].idxmax(), "cv_average_precision_se"]

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.axhspan(best - se, best, color=C[best_model], alpha=0.12, label="within 1 SE of best")
ax.plot(gb["k"], gb["cv_average_precision_mean"], color=C[best_model], marker="o", ms=5, lw=2)
ax.plot(sel["k"], sel["cv_average_precision_mean"], marker="*", ms=18, color="#1a1a1a",
        ls="none", label=f"selected (k={sel['k']})")
ax.set_xlabel("k"); ax.set_ylabel("CV average precision")
ax.set_title(f"1-SE rule on {best_model}: simplest spec inside the band wins", fontsize=10)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
print(f"selected : {sel['model']}  k={sel['k']}  ({sel['rule']})")
print(f"variables: {', '.join(sel['features'])}")
print(f"cost of parsimony vs unconstrained best: {sel['cost_of_parsimony']:.4f} AP, "
      f"saving {sel['unconstrained_best']['k'] - sel['k']} variables")

## Step 7 — holdout confirmation: the honest number

Everything above used only the 75% training partition. The selected spec is now refit and
scored **once** on the untouched 25% holdout — this is the number to quote, because no
part of the selection ever saw these rows. Two readings:

- **CV vs holdout:** if holdout lands near the CV estimate, the process didn't overfit its
  own search. A big shortfall means the leaderboard flattered itself.
- **Decile lift:** operational reading — how concentrated is fraud in the top score bands?
- **Slices:** the same holdout broken down by segment. A strong global AP can hide a big
  flag-rate disparity in one segment — the first thing a fairness/model-risk review asks.

In [ ]:
hm = result.holdout_metrics
print(f"CV estimate {sel['cv_average_precision_mean']:.4f}  →  holdout {hm['average_precision']:.4f}"
      f"   (roc_auc {hm['roc_auc']:.3f}, ks {hm['ks_statistic']:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
d = result.holdout_deciles
axes[0].bar(d["band"], d["lift"], color=C[best_model], width=0.65)
axes[0].axhline(1, color="#888", lw=1)
axes[0].set_xlabel("score decile (1 = highest scores)"); axes[0].set_ylabel("fraud lift vs random")
axes[0].set_title("Holdout gains: fraud concentration by score band", fontsize=10)

s = result.holdout_slices.query("slice_column == 'customer_segment'").sort_values("flag_rate_at_top_pct")
axes[1].barh(s["level"], s["flag_rate_at_top_pct"], color=C[best_model], height=0.55)
for y, (v, n) in enumerate(zip(s["flag_rate_at_top_pct"], s["n"])):
    axes[1].text(v, y, f" {v:.1%} (n={n})", va="center", fontsize=8, color="#555")
axes[1].set_xlabel("share flagged at the review budget")
axes[1].set_title("Same holdout, by customer segment — flag-rate parity check", fontsize=10)
plt.tight_layout()
print("max flag-rate disparity across slice columns:",
      result.report.get("holdout_slices")["max_flag_rate_disparity"], "x")

## Step 8 — the prediction store: every score the harness computed, kept

`run.save_predictions: "all"` told the harness to keep the row-level evidence behind every
number above: for each CV fold of every model × k cell (and for the holdout), one row of
`(row_id, y_true, y_score)` plus the provenance that makes it analysable. Two schema
decisions to understand:

- **Scores, not labels.** Selection never picks a probability threshold — every metric
  above either integrates over *all* thresholds (AP, ROC-AUC, KS) or fixes an *operating
  point* (an FPR budget, a review capacity) and lets the threshold fall out. Storing raw
  scores keeps every operating point computable forever; storing labels would bake one in.
- **`y_true` travels with the score**, so nothing below needs the raw data or the model.

`predictions_meta.json` records what the numbers *mean* — the resolved positive label, the
score semantics, the operating points in force, and the derived decision threshold (Step
10). The store is also on the result object (`result.predictions`); we load from disk here
to make the point that post-run analysis needs only the artifact folder.

In [ ]:
from dmf.research import (load_predictions, load_fold_assignments,
                          compute_metrics, threshold_at_fpr, operating_point_table)

preds, meta = load_predictions("./artifacts_demo/walkthrough")
folds = load_fold_assignments("./artifacts_demo/walkthrough")

print(preds.groupby("stage").size().rename("rows").to_string())
print("\nrow identity  :", meta["row_id_source"])
print("positive label:", meta["positive_label"], "→", meta["score_semantics"])
print(f"decision threshold: {meta['decision_threshold']}  ({meta['decision_threshold_policy']} policy)")
preds.head(4)

## Step 9 — metrics on demand: recompute, expand, drill down

`compute_metrics` runs the stored rows through the **same metric registry the harness
used**, so start with the trust check: per-fold AP from the store, averaged per (model, k),
must reproduce the leaderboard exactly. Then the payoff — metrics **this run never
computed**: the config asked for four, but the store supports the whole registry, at any
operating point, at any granularity (`by=` controls the drill-down). A metric added to
`dmf.metrics` next year will work on this run's store unchanged.

In [ ]:
cv_preds = preds[preds["stage"] == "cv"]
per_fold = compute_metrics(cv_preds, meta, by=["model", "k", "repeat", "fold"])

check = (per_fold.groupby(["model", "k"])["average_precision"].mean().reset_index()
         .merge(lb[["model", "k", "cv_average_precision_mean"]], on=["model", "k"]))
print("store reproduces the leaderboard:",
      np.allclose(check["average_precision"], check["cv_average_precision_mean"], atol=2e-6))

# metrics the run was never configured to compute -- derived from the store alone
from dmf.config import MetricsConfig
expanded = MetricsConfig(
    primary="average_precision",
    secondary=["roc_auc", "ks_statistic", "recall_at_fpr", "lift_at_top_pct",
               "brier_score", "log_loss", "calibration_error"],
    recall_at_fpr=0.01, lift_top_pct=0.05,
)
compute_metrics(preds[preds["stage"] == "holdout"], expanded, by=["model", "k"]).round(4)

**Drill-down: the uncertainty the leaderboard summarised away.** Each dot below is one
fold's AP; the line joins the fold means (the leaderboard values). **How to read it:** a
spec whose dots sit tightly together generalises consistently. A spec with one high fold
and two low ones is being carried by a lucky partition — its leaderboard mean flatters it,
and the corrected SE from Step 4 is the honest width of exactly this spread.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for model, g in per_fold.groupby("model"):
    means = g.groupby("k")["average_precision"].mean()
    ax.plot(means.index, means.values, color=C[model], lw=2, label=model, zorder=2)
    x = g["k"] + (g["fold"].astype(int) - 1) * 0.09          # de-overlap the 3 folds
    ax.scatter(x, g["average_precision"], color=C[model], s=22, alpha=0.45,
               edgecolors="none", zorder=3)
ax.set_xlabel("k (number of variables)"); ax.set_ylabel("fold-level average precision")
ax.set_title("Behind the leaderboard: per-fold AP from the prediction store", fontsize=10)
ax.legend(frameon=False)
plt.tight_layout()

**Overfit, recomputed at will.** The `all` capture level also stored each fold's
*training-side* predictions (`stage == "cv_train"`), so the train-vs-validation gap is
rebuildable per fold and per k without refitting anything. **How to read it:** the
vertical distance between the curves is the generalisation gap; a gap that widens as k
grows means added variables are being memorised, not learned. The leaderboard's
`overfit_gap` column is this same quantity — now yours to slice any way a review asks.

In [ ]:
tr = compute_metrics(preds.query("stage == 'cv_train' and model == @best_model"),
                     meta, by=["k", "fold"])
va = per_fold.query("model == @best_model")
tr_m = tr.groupby("k")["average_precision"].mean()
va_m = va.groupby("k")["average_precision"].mean()

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(tr_m.index, tr_m.values, color=C[best_model], lw=2, ls="--", alpha=0.55)
ax.plot(va_m.index, va_m.values, color=C[best_model], lw=2)
ax.fill_between(va_m.index, va_m.values, tr_m.values, color=C[best_model], alpha=0.08)
ax.text(tr_m.index[-1] + 0.1, tr_m.iloc[-1], "train", color=C[best_model], alpha=0.7,
        fontsize=9, va="center")
ax.text(va_m.index[-1] + 0.1, va_m.iloc[-1], "validation", color=C[best_model],
        fontsize=9, va="center")
k_sel = result.selected["k"]
ax.annotate(f"gap at selected k={k_sel}: {tr_m[k_sel] - va_m[k_sel]:.3f}",
            xy=(k_sel, (tr_m[k_sel] + va_m[k_sel]) / 2),
            xytext=(k_sel + 0.7, (tr_m[k_sel] + va_m[k_sel]) / 2),
            fontsize=8, color="#555",
            arrowprops=dict(arrowstyle="-", color="#999", lw=0.8))
ax.set_xlim(right=tr_m.index.max() + 1.1)
ax.set_xlabel("k"); ax.set_ylabel("average precision")
ax.set_title(f"{best_model}: generalisation gap by k, rebuilt from stored rows", fontsize=10)
plt.tight_layout()

## Step 10 — thresholds: from operating points to the production cut

Nothing above ever chose a probability threshold — yet Step 7 reported FPR-based metrics.
The trick is inversion: training **fixes the operating point** (an FPR budget, a review
capacity) and the threshold *falls out* of the score distribution. `roc_curve` computes
that implied threshold internally and normally discards it; from the store it is
recoverable at any operating point, after the fact:

- **`threshold_at_fpr`** — the score cut achieving a target FPR, **per fold**. The
  fold-to-fold spread (left panel) is the stability evidence to demand before hardcoding a
  production cut: a tight cluster means the operating point is a property of the model, a
  wide one means it is a property of the partition.
- **`operating_point_table`** — precision / recall / flag-rate economics at candidate
  absolute cuts on the holdout (right panel): the sign-off artifact for whoever owns the
  review queue. Moving the cut left buys recall with analyst hours; the curves price that
  trade explicitly.

Expect the two reference lines in the left panel to *disagree*: the fold cluster and the
dotted holdout line mark the **1% FPR** operating point, while the solid decision cut is
the **top-5%-volume** policy — a different operating point gives a different cut, which is
precisely why the choice of policy is explicit configuration rather than a hidden default.

The harness has already derived one cut: `decision_threshold`, via
`metrics.decision_threshold_policy` (default `top_pct` — the holdout score quantile that
flags the top `lift_top_pct` of volume, i.e. a capacity-shaped budget as a *stable
absolute number*).

In [ ]:
ho = preds[preds["stage"] == "holdout"]
k_sel = result.selected["k"]
cell_cv = cv_preds.query("model == @best_model and k == @k_sel")

fold_thr = threshold_at_fpr(cell_cv, max_fpr=0.01, by=["fold"])
ho_thr = threshold_at_fpr(ho, max_fpr=0.01, by=["model"]).loc[0, "threshold"]
cut = meta["decision_threshold"]

MEASURE = {"recall": "#3b6fd4", "precision": "#c8571b", "flag_rate": "#8a5bd6"}
grid = np.unique(np.round(np.quantile(ho["y_score"], np.linspace(0.05, 0.99, 40)), 6))
opt = operating_point_table(ho, thresholds=grid, by=["model"])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1.4]})
ax = axes[0]
ax.scatter(fold_thr["threshold"], np.zeros(len(fold_thr)), s=60, color=C[best_model],
           alpha=0.7, zorder=3, label="CV folds")
ax.axvline(ho_thr, color="#1a1a1a", lw=1.2, ls=":", label=f"holdout ({ho_thr:.3f})")
ax.axvline(cut, color="#1a1a1a", lw=1.6, label=f"decision cut ({cut:.3f})")
ax.set_yticks([]); ax.set_ylim(-1, 1)
ax.set_xlabel("implied threshold at 1% FPR")
ax.set_title("Operating-point stability across folds", fontsize=10)
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
for m, color in MEASURE.items():
    ax.plot(opt["threshold"], opt[m].astype(float), color=color, lw=2,
            label=m.replace("_", " "))
ax.axvline(cut, color="#1a1a1a", lw=1.2, ls="--")
ax.text(cut, 1.03, " decision cut", fontsize=8, color="#1a1a1a")
ax.set_ylim(0, 1.08)
ax.set_xlabel("probability threshold"); ax.set_ylabel("rate")
ax.set_title("Holdout economics per cut: what each threshold buys", fontsize=10)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()

at_cut = operating_point_table(ho, thresholds=[cut], by=["model"]).iloc[0]
print(f"at the bundled cut {cut:.3f}: flag rate {at_cut['flag_rate']:.1%}, "
      f"precision {at_cut['precision']:.1%}, recall {at_cut['recall']:.1%}, "
      f"FPR {at_cut['fpr']:.2%}")

## Step 11 — what leaves the harness

The harness's durable outputs are the winning **specification**, the **prediction store**,
and the fitted **bundle**. `model.joblib` was written with the derived
`decision_threshold` inside, and `ProductionScorer.from_joblib` picks it up — so
production applies the same stable absolute cut Step 10 justified, instead of an arbitrary
0.5 or a batch-relative quantile. (A later cost-based tuning pass overrides it:
`ProductionScorer(..., threshold=...)` wins over the bundle.)

Every scored row still carries a probability *and* a data-quality verdict: rows relying on
out-of-support input are routed to manual review, never auto-actioned.

In [ ]:
scorer = ProductionScorer.from_joblib("./artifacts_demo/walkthrough/model.joblib")
print(f"decision threshold from the bundle: {scorer.threshold:.4f} "
      f"(policy: {meta['decision_threshold_policy']})")

fresh = generate_disputes(n=400, seed=99).drop(columns=["dispute_id", "is_fraudulent_dispute"])
scored, report = scorer.score(fresh)
print("batch verdict:", report["verdict"], "| flag rate:", f"{report['flag_rate']:.1%}")
scored[["fraud_probability", "data_quality", "decision", "action"]].head(5)

## Recap — how each step constrains the next

1. **Config** defines the search space; nothing downstream adds to it.
2. **Ordering (per fold)** decides *which* variable subsets exist — the grid can't score a
   subset the ranking never proposed; stability tells you how trustworthy those subsets are.
3. **Leaderboard** measures every (model, k) honestly; its plateau *suggests* where to stop.
4. **Marginal gains** turn the plateau into per-variable evidence.
5. **1-SE selection** converts that evidence into a single parsimonious spec.
6. **Holdout** is the only number untouched by the search — quote it, and check its slices.
7. **Prediction store** keeps the row-level evidence behind every number above, so any
   metric — configured or not — is recomputable post-run, at any granularity, no refit.
8. **Thresholds** come from operating points: derived on the holdout, stress-tested for
   fold stability, and shipped inside the bundle for the production scorer to pick up.

Full-scale run: `python -m dmf.research.cli train --config ../configs/dispute_fraud.yaml`.
Compare alternative configs honestly: `python -m dmf.research.cli sweep --configs a.yaml b.yaml`.
All tables shown here are written as CSVs under `artifacts_demo/walkthrough/`, alongside
`predictions.parquet`, `fold_assignments.parquet` and `predictions_meta.json` — the store
travels with the run, so every chart in Steps 8–10 can be rebuilt months later from the
artifact folder alone.